In [0]:
data = [
    (101,"Rahul",75000),
    (102,"Priya",85000),
    (103,"Amit",65000)
]

df = spark.createDataFrame(
    data,
    ["emp_id","name","salary"]
)



df.write.format("delta") \
.save("/Volumes/learn_databricks_7405608119894454/default/learn/employees_delta")


delta_df = spark.read.format("delta") \
.load("/Volumes/learn_databricks_7405608119894454/default/learn/employees_delta")

display(delta_df)

emp_id,name,salary
101,Rahul,75000
102,Priya,85000
103,Amit,65000


In [0]:
df.write.format("delta") \
.saveAsTable("employees")

In [0]:
%sql
select * from employees;

emp_id,name,salary
101,Rahul,75000
102,Priya,85000
103,Amit,65000


In [0]:
%sql
CREATE TABLE employees_delta
(
    emp_id INT,
    name STRING,
    salary INT
)
USING DELTA

In [0]:
%sql
INSERT INTO employees_delta
VALUES
(101,'Rahul',75000),
(102,'Priya',85000)

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
select * from employees_delta

emp_id,name,salary
101,Rahul,75000
102,Priya,85000


In [0]:
updates = [

(102,"Priya",90000),
(104,"Sneha",70000)

]

updates_df = spark.createDataFrame(
    updates,
    ["emp_id","name","salary"]
)

In [0]:

from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(
    spark,
    "/Volumes/learn_databricks_7405608119894454/default/learn/employees_delta"
)

delta_table.alias("target") \
.merge(
    updates_df.alias("source"),
    "target.emp_id = source.emp_id"
) \
.whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql DESCRIBE HISTORY employees

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-06-17T09:40:30.000Z,146722045516591,azuser7218_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3731527322015415),85138e50-f2a0-4e82-9597-d2b49bb8ec47,0617-093035-kghxfbj-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1307)",null,Databricks-Runtime/18.2.x-photon-scala2.13


In [0]:
%sql DESCRIBE HISTORY employees_delta

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-06-17T09:44:00.000Z,146722045516591,azuser7218_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3731527322015415),dcdc8c7e-8f64-4e0a-a658-19ceb0808072,0617-093035-kghxfbj-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 1234)",null,Databricks-Runtime/18.2.x-photon-scala2.13
0,2026-06-17T09:43:48.000Z,146722045516591,azuser7218_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-955dd6e9-f1db-4dc7-a71c-7ad5683c58ed"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-abfe80fd-d710-4db3-85ec-0ab23601be6e""}, statsOnLoad -> false)",null,List(3731527322015415),0415d02b-476c-4af8-9b6e-3530cdddcfbb,0617-093035-kghxfbj-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.2.x-photon-scala2.13


In [0]:
df = spark.read.format("delta") \
.option("versionAsOf",0) \
.load("/Volumes/learn_databricks_7405608119894454/default/learn/employees_delta")

display(df)
 

emp_id,name,salary
101,Rahul,75000
102,Priya,85000
103,Amit,65000


In [0]:
%sql OPTIMIZE employees

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1781690536448, 1781690537139, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null)"


In [0]:
%sql OPTIMIZE employees
ZORDER BY(emp_id)

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 1307), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1781690944745, 1781690945373, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null)"


In [0]:
%sql VACUUM employees

path
""
